In [1]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
"/content/drive/MyDrive/cyber_fusion_models/MalwareTextDB-1.0 (1).zip"


'/content/drive/MyDrive/cyber_fusion_models/MalwareTextDB-1.0 (1).zip'

In [3]:
!find /content/drive/MyDrive -name "*MalwareTextDB*.zip"


/content/drive/MyDrive/cyber_fusion_models/MalwareTextDB-1.0 (2).zip
/content/drive/MyDrive/cyber_fusion_models/MalwareTextDB-1.0 (1).zip


In [5]:
!cp "/content/drive/MyDrive/cyber_fusion_models/MalwareTextDB-1.0 (1).zip" /content/


In [6]:
!unzip -o "/content/MalwareTextDB-1.0 (1).zip"


Archive:  /content/MalwareTextDB-1.0 (1).zip
  inflating: annotation_guidelines/AnnotationGuidelines_V2.00.pdf  
  inflating: annotation_guidelines/AttributeLabels_V1.01.pdf  
  inflating: annotation_guidelines/readme.txt  
  inflating: data/.DS_Store          
  inflating: __MACOSX/data/._.DS_Store  
  inflating: data/ann+brown/AdversaryIntelligenceReport_DeepPanda_0 (1).txtcbn  
  inflating: data/ann+brown/Aided_Frame_Aided_Direction.txtcbn  
  inflating: data/ann+brown/Alienvault_Scanbox.txtcbn  
  inflating: data/ann+brown/apt28.txtcbn  
  inflating: data/ann+brown/bcs_wp_InceptionReport_EN_v12914.txtcbn  
  inflating: data/ann+brown/CloudAtlas_RedOctober_APT.txtcbn  
  inflating: data/ann+brown/Compromise_Greece_Beijing.txtcbn  
  inflating: data/ann+brown/Cylance_Operation_Cleaver_Report.txtcbn  
  inflating: data/ann+brown/darkhotel_kl_07.11.txtcbn  
  inflating: data/ann+brown/darkhotelappendixindicators_kl.txtcbn  
  inflating: data/ann+brown/fireeye-operation-saffron-rose.txt

In [7]:
!ls /content/data


ann+brown    brown_ext_training_set  plaintext	 signatures
annotations  event_chains	     readme.txt  tokenized


In [8]:
!git clone https://github.com/mitre/cti.git


fatal: destination path 'cti' already exists and is not an empty directory.


In [9]:
import json

attack_file = "cti/enterprise-attack/enterprise-attack.json"

with open(attack_file, "r", encoding="utf-8") as f:
    data = json.load(f)

benign_texts = []
benign_labels = []

allowed_types = [
    "attack-pattern",
    "course-of-action",
    "x-mitre-data-source",
    "mitigation"
]

for obj in data["objects"]:
    if obj.get("type") in allowed_types:
        desc = obj.get("description", "")
        if desc and len(desc) > 200:
            benign_texts.append(desc)
            benign_labels.append(0)

print("Total benign samples:", len(benign_texts))
print("Sample benign text:\n", benign_texts[0][:300])



Total benign samples: 1097
Sample benign text:
 Ensure only valid password filters are registered. Filter DLLs must be present in Windows installation directory (<code>C:\Windows\System32\</code> by default) of a domain controller and/or local computer with a corresponding entry in <code>HKEY_LOCAL_MACHINE\SYSTEM\CurrentControlSet\Control\Lsa\Not


In [14]:
import glob
import os

base_dir = "/content/data"


malware_folders = [
    "malware/MalwareTextDB/data/plaintext",
    "malware/MalwareTextDB/data/ann-brown",
    "malware/MalwareTextDB/data/brown_ext_training_set",
    "malware/MalwareTextDB/data/event_chains",
    "malware/MalwareTextDB/data/signatures",
    "malware/MalwareTextDB/data/tokenized",

    # large real-world malware reports
    "malware/malware_traffic"
]


malware_texts = []
malware_labels = []

for folder in malware_folders:
    folder_path = os.path.join(base_dir, folder)

    if not os.path.exists(folder_path):
        print(f"Skipping missing folder: {folder}")
        continue

    files = (
        glob.glob(folder_path + "/**/*.txt", recursive=True) +
        glob.glob(folder_path + "/**/*.tokens", recursive=True)
    )

    for file in files:
        try:
            with open(file, "r", encoding="utf-8", errors="ignore") as f:
                text = f.read().replace("\n", " ").strip()

                if len(text) > 150:   # same filter you already use
                    malware_texts.append(text)
                    malware_labels.append(1)
        except:
            continue


print("Total malware samples:", len(malware_texts))
print("Sample malware text:\n", malware_texts[0][:300])


Skipping missing folder: malware/MalwareTextDB/data/plaintext
Skipping missing folder: malware/MalwareTextDB/data/ann-brown
Skipping missing folder: malware/MalwareTextDB/data/brown_ext_training_set
Skipping missing folder: malware/MalwareTextDB/data/event_chains
Skipping missing folder: malware/MalwareTextDB/data/signatures
Skipping missing folder: malware/MalwareTextDB/data/tokenized
Total malware samples: 19
Sample malware text:
 2025-08-20 (WEDNESDAY): KONGTUKE ACTIVITY USING "FILEFIX" STYLE CLICKFIX  REFERENCES:  - https://infosec.exchange/@monitorsg/115061260245537096 - https://infosec.exchange/@malware_traffic/115062221525187895 - https://bsky.app/profile/malware-traffic-analysis.net/post/3lwtw7ftl622s  ASSOCIATED DOMAIN


In [19]:
# Use ALL available malware samples
max_malware = len(malware_texts)

# Keep benign slightly higher (realistic) but safe
target_benign = int(1.2 * max_malware)

if len(benign_texts) > target_benign:
    benign_texts  = benign_texts[:target_benign]
    benign_labels = benign_labels[:target_benign]

print("Malware:", len(malware_texts))
print("Benign :", len(benign_texts))


Malware: 181
Benign : 217


In [20]:
# ================================
# Combine datasets
# ================================

texts = malware_texts + benign_texts
labels = malware_labels + benign_labels

print("Total samples:", len(texts))
print("Total labels:", len(labels))

# Sanity check
print("Malware count:", sum(labels))
print("Benign count:", len(labels) - sum(labels))

# Preview
print("\nSample malware text:\n", texts[labels.index(1)][:300])
print("\nSample benign text:\n", texts[labels.index(0)][:300])



Total samples: 398
Total labels: 398
Malware count: 181
Benign count: 217

Sample malware text:
 Scanbox: A Reconnaissance Framework Used with Watering Hole Attacks A few days ago we detected a watering hole campaign in a website owned by one big industrial company.  The website is related to software used for simulation and system engineering in a wide range of industries, including automotive

Sample benign text:
 Ensure only valid password filters are registered. Filter DLLs must be present in Windows installation directory (<code>C:\Windows\System32\</code> by default) of a domain controller and/or local computer with a corresponding entry in <code>HKEY_LOCAL_MACHINE\SYSTEM\CurrentControlSet\Control\Lsa\Not


In [21]:
# ================================
#  Shuffle dataset
# ================================

import random

combined = list(zip(texts, labels))
random.shuffle(combined)

texts, labels = zip(*combined)

texts = list(texts)
labels = list(labels)

print("Shuffling done.")
print("First 10 labels after shuffle:", labels[:10])


Shuffling done.
First 10 labels after shuffle: [0, 0, 0, 1, 0, 1, 0, 0, 0, 1]


**Train / Validation / Test Split**

In [22]:
# ================================
# Train / Test split
# ================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    texts, labels,
    test_size=0.30,        # 70% train, 30% test
    random_state=42,
    stratify=labels
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

# Label distribution check
print("\nTrain malware:", sum(y_train), "benign:", len(y_train) - sum(y_train))
print("Test malware:", sum(y_test), "benign:", len(y_test) - sum(y_test))



Train size: 278
Test size: 120

Train malware: 126 benign: 152
Test malware: 55 benign: 65


**Tokenization**

In [23]:
!pip install transformers


In [24]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

MAX_LEN = 256 # standard for BERT

def tokenize_texts(texts):
    return tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt"
    )

# Tokenize splits
train_encodings = tokenize_texts(X_train)
test_encodings  = tokenize_texts(X_test)

print("Tokenization done.")
print("Train input_ids shape:", train_encodings["input_ids"].shape)
print("Test input_ids shape:", test_encodings["input_ids"].shape)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Tokenization done.
Train input_ids shape: torch.Size([278, 256])
Test input_ids shape: torch.Size([120, 256])


**Define & Train BERT Classifier**

In [25]:
# Step 1:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertForSequenceClassification
from torch.optim import AdamW


In [26]:
#Step 2: Create a Dataset class (required by PyTorch)
# This just wraps your tokenized data so BERT can train on it.

class TextDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item


In [27]:
#Step 3: Create Train & Validation datasets
train_dataset = TextDataset(train_encodings, y_train)
test_dataset  = TextDataset(test_encodings, y_test)


In [28]:
#Step 4: Create DataLoaders

# ================================

BATCH_SIZE = 8

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


In [29]:
#Step 5: Load pretrained BERT classifier
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2,
    hidden_dropout_prob=0.3,
    attention_probs_dropout_prob=0.3
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.3, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.3, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [30]:
# Define optimizer
optimizer = AdamW(model.parameters(), lr=1e-5)

In [31]:
# Train the model
EPOCHS = 2

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch in train_loader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        total_loss += loss.item()
        loss.backward()
        optimizer.step()

    avg_train_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} - Train Loss: {avg_train_loss:.4f}")


KeyboardInterrupt: 

**Test Evaluation + Probability Extraction**

In [ ]:
# STEP: Test DataLoader

test_dataset = TextDataset(test_encodings, y_test)
test_loader = DataLoader(test_dataset, batch_size=8)


In [ ]:
import torch
import numpy as np

model.eval()

all_labels = []
all_preds  = []
all_probs  = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits
        probs = torch.softmax(logits, dim=1)        # probabilities
        preds = torch.argmax(probs, dim=1)           # predicted class

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs[:, 1].cpu().numpy()) # malware probability

print("Test inference completed.")



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    roc_curve
)

# ===============================
# METRIC COMPUTATION
# ===============================
acc  = accuracy_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds, zero_division=0)
rec  = recall_score(all_labels, all_preds, zero_division=0)
f1   = f1_score(all_labels, all_preds, zero_division=0)
auc  = roc_auc_score(all_labels, all_probs)

print("===== TEXT BRANCH EVALUATION =====")
print(f"Accuracy  : {acc:.4f}")
print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")
print(f"F1-score  : {f1:.4f}")
print(f"ROC-AUC   : {auc:.4f}")

# ===============================
# CONFUSION MATRIX PLOT
# ===============================
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(5,4))
plt.imshow(cm)
plt.title("Confusion Matrix – Text Branch")
plt.colorbar()

plt.xticks([0,1], ["Benign", "Malware"])
plt.yticks([0,1], ["Benign", "Malware"])

for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j],
                 ha="center", va="center")

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()

# ===============================
# ROC CURVE PLOT
# ===============================
fpr, tpr, _ = roc_curve(all_labels, all_probs)

plt.figure(figsize=(5,4))
plt.plot(fpr, tpr, label=f"ROC Curve (AUC = {auc:.3f})")
plt.plot([0,1], [0,1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve – Text Branch")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

# ===============================
# METRIC BAR CHART (OPTIONAL)
# ===============================
metrics = ["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"]
values  = [acc, prec, rec, f1, auc]

plt.figure(figsize=(6,4))
plt.bar(metrics, values)
plt.ylim(0, 1.05)
plt.title("Evaluation Metrics – Text Branch")
plt.ylabel("Score")
plt.tight_layout()
plt.show()


**fusion outputs**

In [ ]:
import numpy as np
import os

os.makedirs("fusion_outputs", exist_ok=True)

np.save("fusion_outputs/text_labels.npy", np.array(all_labels))
np.save("fusion_outputs/text_preds.npy", np.array(all_preds))
np.save("fusion_outputs/text_probs.npy", np.array(all_probs))

print("Text branch outputs saved for fusion.")


**Wrapper**

In [ ]:
def text_branch_predict(raw_text):
    enc = tokenizer(
        raw_text,
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors="pt"
    )

    enc = {k: v.to(device) for k, v in enc.items()}

    model.eval()
    with torch.no_grad():
        outputs = model(**enc)
        probs = torch.softmax(outputs.logits, dim=1)

    return probs[0, 1].item()


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, json

BASE = "/content/drive/MyDrive/cyber_fusion_models"
TEXT_DIR = os.path.join(BASE, "text_branch_artifact")
os.makedirs(TEXT_DIR, exist_ok=True)

# Save trained model + tokenizer
model.save_pretrained(TEXT_DIR)
tokenizer.save_pretrained(TEXT_DIR)

print("Text model and tokenizer saved to:", TEXT_DIR)


In [ ]:
meta = {
    "max_len": 512,        # must match training
    "threshold": 0.5,      # or tuned threshold if you used one
    "positive_class": 1,   # malware = 1
    "model_type": "transformer"
}

with open(os.path.join(TEXT_DIR, "meta.json"), "w") as f:
    json.dump(meta, f, indent=2)

print("Text inference metadata saved")


In [ ]:
import pandas as pd
from google.colab import drive
import os

drive.mount("/content/drive")

SAVE_DIR = "/content/drive/MyDrive/cyber_fusion_models/results"
os.makedirs(SAVE_DIR, exist_ok=True)

text_results = pd.DataFrame({
    "id": np.arange(len(all_labels)),
    "y_true": all_labels,
    "prob": all_probs
})

text_results.to_csv(f"{SAVE_DIR}/text_results.csv", index=False)
print("Text results saved.")
